# Day-to-day evolution of (ridepooling) supply and demand
Module for simulating ridesourcing evolution, including pooled rides

Contribution by Arjan de Ruijter - a.j.f.deruijter@tudelft.nl

In [2]:
%load_ext autoreload
%autoreload 2
import os, sys # add MaaSSim and MaaSSim/MaaSSim to path (not needed if already in path)
module_path = os.path.abspath(os.path.join('../..'))
if module_path not in sys.path:
    sys.path.append(module_path)

In [3]:
from MaaSSim.utils import save_config, get_config, load_G, generate_demand, initialize_df, empty_series, \
    slice_space, test_space, read_requests_csv
from MaaSSim.maassim import Simulator
from MaaSSim.data_structures import structures as inData
from MaaSSim.d2d_sim import *
from MaaSSim.d2d_demand import *
from MaaSSim.d2d_supply import *
from MaaSSim.d2d_shared import prep_shared_rides
from MaaSSim.decisions import dummy_False
from MaaSSim.exmas import main

In [4]:
import pandas as pd
import zipfile
import logging
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from matplotlib.lines import Line2D
import numpy as np
import random
# import ExMAS
plt.style.use('ggplot')
np.random.seed(0)
random.seed(0)

In [5]:
# Load config
params = get_config('../../data/config/delft.json')  # load configuration
params.paths.albatross = '../../data/albatross'

In [6]:
# Experiment replications and number of threads to be used
params.parallel.nReplications = 1
params.parallel.nThread = 1

# Main experimental settings
params.nP = 500 # travellers
params.nV = 50 # drivers
params.nD = 3 # days
params.simTime = 2 # hours

In [7]:
# Other day-to-day settings
params.evol.drivers.kappa = 0.2 # learning weight (supply-side)
params.evol.drivers.res_wage.mean = 25 #euros/h
params.evol.drivers.gini = 0.35 # gini coefficient used to establish sigma parameter of log-norm distribution of res wage
params.evol.drivers.init_inc_ratio = 1 #expected income of informed drivers at start of sim as ratio of res wage

params.evol.drivers.inform.prob_start = 1 # probability of being informed at start of sim
params.evol.drivers.inform.beta = 0.1 # information transmission rate
params.evol.drivers.inform.std_fact = 0.5 # multiplier of the standard deviation of experienced income used in signal

params.evol.drivers.regist.prob_start = 1 # probability of being registered if informed at start of sim
params.evol.drivers.regist.beta = 0.2 # registration choice model parameter
params.evol.drivers.regist.cost_comp = 20 # daily share of registration costs (euros)
params.evol.drivers.regist.samp = 0.5 # probability of making (de)regist decision
params.evol.drivers.regist.min_work_exp = 0 # Working experience required before deregistration is possible
params.evol.drivers.regist.min_days = 5 # Minimum number of registered days before driver can deregister

params.evol.drivers.particip.beta = 0.1 # participation choice model parameter
params.evol.drivers.particip.probabilistic = True # stochasticity in participation choice

params.evol.travellers.inform.prob_start = 1 # probability that traveller is informed at start of sim
params.evol.travellers.inform.beta = 0.1 # information transmission rate (demand-side)
params.evol.travellers.inform.start_wait = 0 # expected waiting time at start of simulation
params.evol.travellers.inform.std_fact = 0.5 # multiplier of the standard deviation of experienced waiting time used in signal
params.evol.travellers.reject_penalty = 30 * 60 # seconds
params.evol.travellers.kappa = 0.2 # learning weight (demand-side)
params.evol.travellers.min_prob = 0.05 # filtering criterion, when probability is lower when waiting time is zero, never consider RS

params.evol.travellers.mode_pref.mean_vot = 10 # Mean VoT in euro/h
params.evol.travellers.mode_pref.access_multip = 2 # Multiplier of access time compared to in-vehicle time
params.evol.travellers.mode_pref.wait_multip = 2.5 # Multiplier of waiting time compared to in-vehicle time
params.evol.travellers.mode_pref.bike_multip = 2 # Multiplier of biking time compared to in-vehicle time
params.evol.travellers.mode_pref.beta_cost = -0.1592 # util/euro
params.evol.travellers.mode_pref.transfer_pen = 5 * 60 # seconds, to be added to IVT for each transfer
params.evol.travellers.mode_pref.ASC_car = 0 # util, rel to bike
params.evol.travellers.mode_pref.ASC_rs = 0
params.evol.travellers.mode_pref.ASC_pt =  0
params.evol.travellers.mode_pref.ASC_car_sd = 0 # standard deviation in ASCs
params.evol.travellers.mode_pref.ASC_rs_sd = 0
params.evol.travellers.mode_pref.ASC_pt_sd = 0
params.evol.travellers.mode_pref.ASC_bike_sd = 0
params.evol.travellers.mode_pref.gini = params.evol.drivers.gini

# Financial settings
params.platforms.base_fare = 1 #euro
params.platforms.fare = 1 #euro/km
params.platforms.min_fare = 0 # euro
params.platforms.comm_rate = 0.05 #rate
params.drivers.fuel_costs = 0.25 #euro/km

# Properties alternative modes
params.alt_modes.pt.option = False
params.alt_modes.pt.base_fare = 0.99 # euro
params.alt_modes.pt.km_fare = 0.174 # euro/km
params.alt_modes.car.km_cost = 0.5 # euro/km
params.alt_modes.car.diff_parking = True # different parking tariffs in city
params.alt_modes.car.park_cost = 7.5 # euro
params.alt_modes.car.park_cost_center = 15 # euro
params.alt_modes.car.access_time = 10 * 60 # s
params.speeds.bike = (1/2.5) * params.speeds.ride # m/s

# Regulation
params.platforms.reg_cap = np.inf # registration cap
params.platforms.ptcp_cap = np.inf # daily participation cap

# Demand settings
# params.demand_structure.origins_dispertion = -0.0003
# params.demand_structure.destinations_dispertion = -0.0003
params.dist_threshold_min = 2000 # min dist
# params.dist_threshold = 100000 # max dist

# Start time
# params.t0 = pd.Timestamp.now()
params.t0 = pd.Timestamp(2021, 11, 1, 9)

In [8]:
# Pooling settings
params.shareability.offered = True
params.shareability.min_discount = 0.3 # fraction of solo fare
params.shareability.add_discount = 0.3 # fraction of solo fare
params.shareability.shared_discount = 0.35 # as considered by platform in determ. of feasible rides (fraction of solo fare)
params.shareability.delay_value = 1
params.shareability.WtS_mean = 1.5 # Actual (mean) WtS multipl. in population
params.shareability.WtS = 1 # Willingness to share multipl. as considered by platform in determining feasible rides
params.shareability.WtS_std = 0 # St. dev. of WtS in population
params.shareability.matching_obj = 'u_pax' #minimize VHT for vehicles
params.shareability.pax_delay = 0 # additional delay for picking up travs
params.shareability.max_delay = 10 * 60 # max. allowed delay in pooled rides (sec.)
params.shareability.horizon = 600
params.shareability.max_degree = 2
params.shareability.share = 1
params.shareability.without_matching = True

In [9]:
# Compute (other) input parameters in ExMAS
params.shareability.avg_speed = params.speeds.ride
params.shareability.price = params.platforms.fare #eur/km
params.shareability.nP = params.nP
params.shareability.VoT = params.evol.travellers.mode_pref.mean_vot / 3600 # convert eur/h to eur/s

In [10]:
inData = load_G(inData, params, stats=True, set_t=False)  # download graph for the 'params.city' and calc the skim matrices
if params.alt_modes.car.diff_parking:
    inData = diff_parking(inData) # determine which nodes are in center

In [11]:
inData = generate_demand(inData, params, avg_speed = True)

In [12]:
# Load processed Albatross file, the OTP result, and compute PT fares
# inData = load_albatross_proc(inData, params, avg_speed = True)
# inData.requests = inData.requests.drop(['orig_geo', 'dest_geo', 'origin_y', 'origin_x', 'destination_y', 'destination_x', 'time'], axis = 1)
# inData.pt_itinerary = load_OTP_result(params)
# inData = consist_OTP_alba(inData, params)

In [13]:
# Prepare supply and demand attributes
inData.passengers = prefs_travs(inData, params)
all_pax = mode_filter(inData, params)
inData.passengers = all_pax[all_pax.mode_choice == "day-to-day"]
inData.requests = inData.requests[inData.requests.pax_id.isin(inData.passengers.index)]
if params.alt_modes.pt.option:
    inData.pt_itinerary = inData.pt_itinerary[inData.pt_itinerary.pax_id.isin(inData.passengers.index)]
    inData.passengers.reset_index(drop=True, inplace=True)
    inData.requests.reset_index(drop=True, inplace=True)
    inData.pt_itinerary.reset_index(drop=True, inplace=True)
inData.requests['pax_id'] = inData.requests.index
inData.pt_itinerary['pax_id'] = inData.pt_itinerary.index
inData.passengers['informed'] = np.random.rand(len(inData.passengers)) < params.evol.travellers.inform.prob_start
inData.passengers['expected_wait'] = params.evol.travellers.inform.start_wait
inData.passengers['expected_wait_pool'] = params.evol.travellers.inform.start_wait
inData.passengers['expected_pool_disc'] = params.shareability.min_discount
inData.passengers['expected_pool_detour'] = 0
fixed_supply = generate_vehicles_d2d(inData, params)
inData.vehicles = fixed_supply.copy()
inData.vehicles.platform = inData.vehicles.apply(lambda x: 0, axis = 1)
inData.passengers.platforms = inData.passengers.apply(lambda x: [0], axis = 1)
inData.requests['platform'] = inData.requests.apply(lambda row: inData.passengers.loc[row.name].platforms[0], axis = 1) 
inData.platforms = pd.concat([inData.platforms,pd.DataFrame(columns=['base_fare','comm_rate','min_fare'])])
inData.platforms = initialize_df(inData.platforms)
inData.platforms.loc[0]=[params.platforms.fare,'Uber',30,params.platforms.base_fare,params.platforms.comm_rate,params.platforms.min_fare,]

In [14]:
inData = main(inData, params.shareability, plot=False) # create shareability graph (ExMAS) 

08-05-23 14:12:22-INFO-Initializing pairwise trip shareability between 500 and 500 trips.
08-05-23 14:12:22-INFO-creating combinations
08-05-23 14:12:22-INFO-249500	 nR*(nR-1)
08-05-23 14:12:28-INFO-Reduction of feasible pairs by 97.90%
08-05-23 14:12:28-INFO-Degree 2 	Completed


In [15]:
# Day-to-day simulation (incl. processing)
sim = Simulator(inData, params=params,
                    kpi_veh = D2D_veh_exp,
                    kpi_pax = d2d_kpi_pax,
                    f_driver_out = D2D_driver_out,
                    f_trav_out = d2d_no_request,
                    f_trav_mode = dummy_False,
                    logger_level=logging.WARNING)  # initialize

evol_micro = init_d2d_dotmap(params)
plf_stats = pd.DataFrame(columns = ['profit','veh_dist','pax_dist'])
for day in range(params.get('nD', 1)):  # run iterations
    inData.passengers = mode_preday(inData, params)
    temp_rides = inData.sblts.rides.copy()
    temp_reqs = inData.sblts.requests.copy()
    
    # Determine mode choice on this day
    rs_users = inData.passengers[(inData.passengers.mode_day == 'rs') | (inData.passengers.mode_day == 'pool')].index.tolist()
    poolers = inData.passengers[inData.passengers.mode_day == 'pool'].index.tolist()
    inData.sblts.rides = inData.sblts.rides[inData.sblts.rides.apply(lambda x: all(i in rs_users for i in x.indexes), axis=1)] # filter out all travellers opting for mode outside ride-hailing market
    inData.sblts.rides = inData.sblts.rides[inData.sblts.rides.apply(lambda x: (all(i in poolers for i in x.indexes) or x.kind == 1), axis=1)]  # filter out pooled trips for individuals opting for private ride
    inData.sblts.requests = inData.sblts.requests[inData.sblts.requests.apply(lambda x: x.pax_id in rs_users, axis=1)]
    
    inData = prep_shared_rides(inData, params.shareability)  # prepare schedules
    
    # Determine which drivers participate on this day
    inData.vehicles.rand_val = np.random.rand(len(inData.vehicles))
    inData.vehicles.ptcp_day = inData.vehicles.apply(lambda x: bool((np.exp(params.evol.drivers.particip.beta * x.expected_income) / 
                                                     (np.exp(params.evol.drivers.particip.beta * x.expected_income) + 
                                                      np.exp(params.evol.drivers.particip.beta * x.res_wage)) * x.registered) < x.rand_val), axis=0)
    
    # Run FleetPy

    
    # Update statistics
    drivers_summary = update_d2d_drivers(sim=sim,params=params)
    travs_summary = update_d2d_travellers(sim=sim,params=params)
    
    exp_df = update_work_exp(inData, drivers_summary)
    inData.vehicles.work_exp = exp_df.work_exp
    inData.vehicles.days_since_reg = exp_df.days_since_reg

    res_inf_driver = wom_driver(inData, params = params)
    inData.vehicles.informed = res_inf_driver
    inData.vehicles.expected_income = learning_unregist(inData, drivers_summary, params = params)
    
    res_regist = platform_regist(inData, drivers_summary, params = params)
    inData.vehicles.registered = res_regist.registered
    inData.vehicles.work_exp = res_regist.work_exp
    inData.vehicles.pos = fixed_supply.pos
    inData.vehicles.rejected_reg = res_regist.rejected_reg
    
    res_inf_trav = wom_trav(inData, travs_summary, params = params)
    inData.passengers.informed = res_inf_trav.informed
    inData.passengers.expected_wait = res_inf_trav.perc_wait
    inData.passengers.expected_wait_pool = res_inf_trav.perc_wait_pool
    inData.passengers.expected_pool_disc = res_inf_trav.perc_disc
    inData.passengers.expected_pool_detour = res_inf_trav.perc_detour
    
    inData.sblts.rides = temp_rides.copy()
    inData.sblts.requests = temp_reqs.copy()
    
    evol_micro = d2d_summary_day(evol_micro, drivers_summary, travs_summary, day)
    evol_micro = d2d_summ_pooling(evol_micro, travs_summary)
    row = {'profit': sim.last_res.veh_kpi.REVENUE.loc['sum'] / (1 - params.platforms.comm_rate), 'veh_dist': sim.last_res.veh_kpi.DRIVING_DIST.loc['sum'], 'pax_dist': sim.last_res.pax_kpi.TRAVEL.loc['sum'] * params.speeds.ride / 1000}
    plf_stats = plf_stats.append(row, ignore_index=True)

08-05-23 14:12:29-WARNING-Setting up 2h simulation at 2021-11-01 08:00:09 for 50 vehicles and 500 passengers in Delft, Netherlands
08-05-23 14:12:47-WARNING-day 0: simulation time 14.3 s
08-05-23 14:12:47-WARNING-assertion tests for simulation results - passed
08-05-23 14:13:00-WARNING-day 1: simulation time 9.8 s
08-05-23 14:13:01-WARNING-assertion tests for simulation results - passed
08-05-23 14:13:11-WARNING-day 2: simulation time 8.2 s
08-05-23 14:13:12-WARNING-assertion tests for simulation results - passed


In [16]:
evol_micro, evol_agg = d2d_agg_statistics(evol_micro, params) # multi-day stats

In [17]:
# Save d2d stats to zip file
with zipfile.ZipFile('evol.zip', 'w') as csv_zip:
    csv_zip.writestr("evol_agg_supply.csv", evol_agg.supply.to_csv())
    csv_zip.writestr("evol_agg_demand.csv", evol_agg.demand.to_csv())

In [18]:
evol_agg.supply

,inform,regist,rejected_reg,particip,reject_particip,mean_perc_inc,mean_perc_inc_ptcp,mean_perc_inc_reg,mean_exp_inc
day,,,,,,,,,
0,50,50,0,28,0,50.539889,51.872048,50.539889,26.425607
1,50,50,0,21,0,47.689888,49.804478,47.689888,22.544857
2,50,50,0,26,0,45.400079,37.152604,45.400079,14.275404


In [19]:
evol_agg.demand

,inform,requests,req_solo,req_pool,gets_offer_solo,gets_offer_pooling,act_shared,mean_wait_solo,corr_mean_wait_solo,mean_wait_pooling,...,perc_wait,perc_wait_req,perc_wait_pool,mean_disc,perc_disc,mean_detour,perc_detour,bike,car,pt
day,,,,,,,,,,,,,,,,,,,,,
0,500,304,135,169,135,169,166,188.200658,188.200658,120.986842,...,0.0000,0.0,0.000000,0.594675,0.300000,46.460526,0.00000,162,34,0
1,500,177,113,64,113,64,58,175.474576,175.474576,77.254237,...,8.1732,0.0,20.153425,0.571875,0.319920,39.033898,5.64960,286,37,0
2,500,135,86,49,86,49,46,120.548148,120.548148,57.859259,...,15.1272,0.0,27.520159,0.581633,0.326736,45.948148,8.40952,331,34,0


In [20]:
# Plot evolution of main performance indicators
fig, axes = plt.subplots(nrows=9, ncols=1, figsize = (10,20), sharex = True)
evol_agg.supply[['inform','regist','particip']].plot(ax = axes[0], color=['lightsteelblue','tab:blue','midnightblue'])
axes[0].set_title('(A) Ridesourcing supply')
axes[0].legend(['Informed','Registered','Participating'])
axes[0].set_ylim([0,params.nV + 25])
axes[0].set_ylabel('Number of drivers')
evol_agg.supply[['mean_perc_inc','mean_exp_inc']].plot(ax = axes[2], color=['lightsteelblue','midnightblue'])
axes[2].set_title('(C) Driver earnings')
axes[2].legend(['Expected','Experienced'])
axes[2].set_ylim([0,math.ceil(max(evol_agg.supply.mean_perc_inc.max(),evol_agg.supply.mean_exp_inc.max())/50)*50])
axes[2].set_ylabel('Income (\u20ac)')

evol_agg.demand[['req_solo','req_pool','bike','car','pt']].plot.area(ax = axes[1])
h,l = axes[1].get_legend_handles_labels()
evol_agg.demand['inform'].plot(ax = axes[1], color = 'black', linestyle = 'dashed', label = 'informed')
line = Line2D([0], [0],color='black', linestyle ='dashed')
axes[1].set_title('(B) Demand')
axes[1].set_ylim([0,len(inData.passengers) * 1.1])
axes[1].set_ylabel('Number of travellers')
h.extend([line])
axes[1].legend(labels=["Solo RS","Solo pool","Bike","Car","Public transport","Informed"], handles=h)

evol_agg.demand['mean_wait_solo'].apply(lambda x: 1/60 * x).plot(ax = axes[3], label='Experienced - solo')
# evol_agg.demand['corr_mean_wait_solo'].apply(lambda x: 1/60 * x).plot(ax = axes[3], label='Corrected exp. - solo')
evol_agg.demand['perc_wait'].apply(lambda x: 1/60 * x).plot(ax = axes[3], label='Expected - solo')
evol_agg.demand['mean_wait_pooling'].apply(lambda x: 1/60 * x).plot(ax = axes[3], label='Experienced - pooled')
# evol_agg.demand['corr_mean_wait_pooling'].apply(lambda x: 1/60 * x).plot(ax = axes[3], label='Corrected exp. - pooled')
evol_agg.demand['perc_wait_pool'].apply(lambda x: 1/60 * x).plot(ax = axes[3], label='Expected - pooled')
axes[3].legend()
axes[3].set_title('(D) Waiting time')
axes[3].set_ylabel('Mean wait. time (min)')

evol_agg.demand['proport_match_solo'] = evol_agg.demand.gets_offer_solo / evol_agg.demand.req_solo * 100
evol_agg.demand['proport_match_pool'] = evol_agg.demand.gets_offer_pooling / evol_agg.demand.req_pool * 100
evol_agg.demand['act_shared_rel'] = evol_agg.demand.act_shared / evol_agg.demand.gets_offer_pooling * 100
evol_agg.demand['proport_match_solo'].plot(ax = axes[4], label='Solo requests returned with offer')
evol_agg.demand['proport_match_pool'].plot(ax = axes[4], label='Pooled requests returned with offer')
evol_agg.demand['act_shared_rel'].plot(ax = axes[4], label='Share of pooled rides with actual sharing', linestyle = 'dotted')
axes[4].legend()
axes[4].set_title('(E) Proportion matched')
axes[4].yaxis.set_major_formatter(mtick.PercentFormatter())

evol_agg.demand['mean_detour'].apply(lambda x: 1/60 * x).plot(ax = axes[5], label='Experienced detour')
evol_agg.demand['perc_detour'].apply(lambda x: 1/60 * x).plot(ax = axes[5], label='Expected detour')
axes[5].legend()
axes[5].set_title('(F) Pooling detour')
axes[5].set_ylabel('Mean detour time (min)')

evol_agg.demand['mean_disc'].apply(lambda x: 100 * x).plot(ax = axes[6], label='Experienced discount')
evol_agg.demand['perc_disc'].apply(lambda x: 100 * x).plot(ax = axes[6], label='Expected discount')
axes[6].legend()
axes[6].set_title('(G) Pooling discount')
axes[6].yaxis.set_major_formatter(mtick.PercentFormatter())

plf_stats['profit'].plot(ax = axes[7], label='Platform profit')
axes[7].set_title('(H) Platform profit')

plf_stats['dist_ratio'] = plf_stats.pax_dist / plf_stats.veh_dist
plf_stats['dist_ratio'].plot(ax = axes[8], label='Pax km per veh km')
axes[8].legend()
axes[8].set_title('(I) Distance savings')

plt.savefig('d2d-evo.png')

---

In [21]:
sim.last_res.veh_exp

,nRIDES,nREJECTED,DRIVING_TIME,DRIVING_DIST,REVENUE,COST,NET_INCOME,OUT,FORCED_OUT,STARTS_DAY,...,ARRIVES_AT_PICKUP,MEETS_TRAVELLER_AT_PICKUP,DEPARTS_FROM_PICKUP,ARRIVES_AT_DROPOFF,CONTINUES_SHIFT,STARTS_REPOSITIONING,REPOSITIONED,DECIDES_NOT_TO_DRIVE,ENDS_SHIFT,NOT_ALLOWED_TO_DRIVE
veh,,,,,,,,,,,,,,,,,,,,,
1,5.0,0.0,2063.0,20.63,20.8525,5.1575,15.6950,False,False,0,...,14.0,150.0,1695.0,11992.0,0,0,0,0,0,0
2,6.0,0.0,2251.0,22.51,23.0470,5.6275,17.4195,False,False,0,...,0.0,180.0,1826.0,10185.0,0,0,0,0,0,0
3,0.0,0.0,0.0,0.00,0.0000,0.0000,0.0000,True,False,0,...,0.0,0.0,0.0,0.0,0,0,0,0,0,0
4,3.0,0.0,1611.0,16.11,12.0080,4.0275,7.9805,False,False,0,...,0.0,90.0,964.0,11882.0,0,0,0,0,0,0
5,7.0,0.0,2811.0,28.11,28.1295,7.0275,21.1020,False,False,0,...,16.0,210.0,2261.0,10107.0,0,0,0,0,0,0
6,4.0,0.0,1504.0,15.04,13.1955,3.7600,9.4355,False,False,0,...,0.0,120.0,989.0,11607.0,0,0,0,0,0,0
7,4.0,0.0,1215.0,12.15,14.4970,3.0375,11.4595,False,False,0,...,51.0,120.0,1126.0,11157.0,0,0,0,0,0,0
8,6.0,0.0,1965.0,19.65,20.5580,4.9125,15.6455,False,False,0,...,0.0,180.0,1564.0,11196.0,0,0,0,0,0,0
9,0.0,0.0,0.0,0.00,0.0000,0.0000,0.0000,True,False,0,...,0.0,0.0,0.0,0.0,0,0,0,0,0,0


In [22]:
sim.last_res.pax_exp

,ACCEPTS_OFFER,ARRIVES_AT_DEST,ARRIVES_AT_DROPOFF,ARRIVES_AT_PICKUP,DEPARTS_FROM_PICKUP,MEETS_DRIVER_AT_PICKUP,PREFERS_OTHER_SERVICE,RECEIVES_OFFER,REQUESTS_RIDE,SETS_OFF_FOR_DEST,NO_REQUEST,OTHER_MODE,STARTS_DAY,IS_REJECTED_BY_VEHICLE,REJECTS_OFFER,LOSES_PATIENCE,TRAVEL,WAIT,OPERATIONS
pax,,,,,,,,,,,,,,,,,,,
0,15.0,0.0,262.0,20.0,30.0,99.0,0.0,0.0,0.0,10.0,False,False,0,0,0,0,262.0,99.0,55.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,True,False,0,0,0,0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,True,False,0,0,0,0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,True,False,0,0,0,0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,True,False,0,0,0,0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,True,False,0,0,0,0,0.0,0.0,0.0
496,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,True,False,0,0,0,0,0.0,0.0,0.0
497,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,True,False,0,0,0,0,0.0,0.0,0.0


In [23]:
inData.sblts.R[1]

,indexes,u_pax,u_veh,kind,u_paxes,times,indexes_orig,indexes_dest
0,[0],3.350778,262,1,[3.350777777777778],"[0, 262]",[0],[0]
1,[1],4.528333,354,1,[4.528333333333333],"[11, 354]",[1],[1]
2,[2],3.085444,241,1,[3.0854444444444447],"[14, 241]",[2],[2]
3,[3],5.088556,398,1,[5.088555555555556],"[29, 398]",[3],[3]
4,[4],2.713889,212,1,[2.713888888888889],"[45, 212]",[4],[4]
...,...,...,...,...,...,...,...,...
495,[495],3.254556,254,1,[3.2545555555555556],"[7109, 254]",[495],[495]
496,[496],3.032333,237,1,[3.0323333333333333],"[7127, 237]",[496],[496]
497,[497],5.865000,459,1,[5.865],"[7133, 459]",[497],[497]
498,[498],6.995444,547,1,[6.995444444444445],"[7140, 547]",[498],[498]


In [24]:
inData.sblts.rides

,indexes,u_pax,u_veh,kind,u_paxes,times,indexes_orig,indexes_dest,index
0,[0],3.350778,262,1,[3.350777777777778],"[0, 262]",[0],[0],0
1,[1],4.528333,354,1,[4.528333333333333],"[11, 354]",[1],[1],1
2,[2],3.085444,241,1,[3.0854444444444447],"[14, 241]",[2],[2],2
3,[3],5.088556,398,1,[5.088555555555556],"[29, 398]",[3],[3],3
4,[4],2.713889,212,1,[2.713888888888889],"[45, 212]",[4],[4],4
...,...,...,...,...,...,...,...,...,...
16860,"[105, 119]",7.527361,602,21,"[4.436233333333334, 3.0911277777777775]","[1661.5, 167, 330, 105]","[105, 119]","[119, 105]",16860
16861,"[105, 108]",7.090650,702,21,"[4.784844444444445, 2.3058055555555557]","[1636.0, 103, 238, 361]","[105, 108]","[108, 105]",16861
16862,"[105, 136]",8.430311,776,21,"[5.340400000000001, 3.0899111111111113]","[1830.0, 112, 285, 379]","[105, 136]","[136, 105]",16862
16863,"[105, 87]",7.899061,575,21,"[4.854288888888889, 3.044772222222222]","[1484.0, 55, 272, 248]","[105, 87]","[87, 105]",16863


In [25]:
inData.sblts.requests

,index,pax_id,origin,destination,treq,tdep,ttrav,tarr,tdrop,shareable,schedule_id,dist,car_park_cost,dest_center,platform,VoT,delta,u,u_PT
0,0,0,1679761158,2612573901,0,NaN,262,2021-11-01 08:04:31,NaN,False,NaN,2623,7.5,False,0,0.002778,330.498,3.350778,999999
1,1,1,44824666,44764023,11,NaN,354,2021-11-01 08:06:14,NaN,False,NaN,3545,7.5,False,0,0.002778,446.670,4.528333,999999
2,2,2,4053473202,1385072823,14,NaN,241,2021-11-01 08:04:24,NaN,False,NaN,2416,7.5,False,0,0.002778,304.416,3.085444,999999
3,3,3,44862225,44759522,29,NaN,398,2021-11-01 08:07:16,NaN,False,NaN,3983,7.5,False,0,0.002778,501.858,5.088556,999999
4,4,4,1679761151,44839603,45,NaN,212,2021-11-01 08:04:26,NaN,False,NaN,2125,7.5,False,0,0.002778,267.750,2.713889,999999
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,495,495,1413911080,2321923795,7109,NaN,254,2021-11-01 10:02:52,NaN,False,NaN,2549,7.5,False,0,0.002778,321.174,3.254556,999999
496,496,496,1421472391,1519889948,7127,NaN,237,2021-11-01 10:02:53,NaN,False,NaN,2374,7.5,False,0,0.002778,299.124,3.032333,999999
497,497,497,44817967,1679761035,7133,NaN,459,2021-11-01 10:06:41,NaN,False,NaN,4590,7.5,False,0,0.002778,578.340,5.865000,999999
498,498,498,44892459,643587012,7140,NaN,547,2021-11-01 10:08:16,NaN,False,NaN,5476,7.5,False,0,0.002778,600.000,6.995444,999999


In [26]:
sim.runs[0].rides

,veh,pos,t,event,paxes
0,21,4.483099e+07,0,STARTS_DAY,[]
1,21,4.483099e+07,0,OPENS_APP,[]
2,21,4.483099e+07,568,RECEIVES_REQUEST,[]
3,21,4.483099e+07,583,ACCEPTS_REQUEST,[]
4,21,4.483099e+07,603,IS_ACCEPTED_BY_TRAVELLER,[]
...,...,...,...,...,...
74,28,1.421472e+09,7277,DEPARTS_FROM_PICKUP,[496]
75,28,1.519890e+09,7514,ARRIVES_AT_DROPOFF,[496]
76,28,1.519890e+09,14399,ENDS_SHIFT,[]
0,1,1.571093e+09,0,STARTS_DAY,[]


In [27]:
sim.runs[0].trips

,pax,pos,t,event,veh_id
0,0,1679761158,0,STARTS_DAY,NaN
1,0,1679761158,0,PREFERS_OTHER_SERVICE,NaN
0,1,44824666,0,STARTS_DAY,NaN
1,1,44824666,11,REQUESTS_RIDE,NaN
2,1,44824666,11,RECEIVES_OFFER,NaN
...,...,...,...,...,...
7,498,643587012,7803,ARRIVES_AT_DROPOFF,39.0
8,498,643587012,7813,SETS_OFF_FOR_DEST,NaN
9,498,643587012,7813,ARRIVES_AT_DEST,NaN
0,499,44856860,0,STARTS_DAY,NaN


In [28]:
inData.sblts.SINGLES

,indexes,u_pax,u_veh,kind,u_paxes,times,indexes_orig,indexes_dest
0,[0],3.350778,262,1,[3.350777777777778],"[0, 262]",[0],[0]
1,[1],4.528333,354,1,[4.528333333333333],"[11, 354]",[1],[1]
2,[2],3.085444,241,1,[3.0854444444444447],"[14, 241]",[2],[2]
3,[3],5.088556,398,1,[5.088555555555556],"[29, 398]",[3],[3]
4,[4],2.713889,212,1,[2.713888888888889],"[45, 212]",[4],[4]
...,...,...,...,...,...,...,...,...
495,[495],3.254556,254,1,[3.2545555555555556],"[7109, 254]",[495],[495]
496,[496],3.032333,237,1,[3.0323333333333333],"[7127, 237]",[496],[496]
497,[497],5.865000,459,1,[5.865],"[7133, 459]",[497],[497]
498,[498],6.995444,547,1,[6.995444444444445],"[7140, 547]",[498],[498]


In [29]:
xyz = inData.sblts.rides[inData.sblts.rides.apply(lambda x: all(i in rs_users for i in x.indexes), axis=1)] # filter out all travellers opting for mode outside ride-hailing market
xyz = xyz[xyz.apply(lambda x: (all(i in poolers for i in x.indexes) or x.kind == 1), axis=1)]  # filter out pooled trips for individuals opting for private ride
xyz
# poolers

,indexes,u_pax,u_veh,kind,u_paxes,times,indexes_orig,indexes_dest,index
0,[0],3.350778,262,1,[3.350777777777778],"[0, 262]",[0],[0],0
5,[5],4.938222,386,1,[4.9382222222222225],"[53, 386]",[5],[5],5
6,[6],3.378333,264,1,[3.3783333333333334],"[54, 264]",[6],[6],6
12,[12],3.954333,309,1,[3.9543333333333335],"[169, 309]",[12],[12],12
14,[14],2.652000,207,1,[2.652],"[171, 207]",[14],[14],14
...,...,...,...,...,...,...,...,...,...
16208,"[409, 420]",6.386200,519,21,"[3.7756555555555553, 2.610544444444445]","[6019.5, 98, 254, 167]","[409, 420]","[420, 409]",16208
16209,"[409, 410]",5.899606,558,21,"[3.777044444444444, 2.122561111111111]","[5877.0, 119, 213, 226]","[409, 410]","[410, 409]",16209
16480,"[413, 452]",9.530578,688,21,"[5.736122222222223, 3.794455555555556]","[6241.0, 81, 340, 267]","[413, 452]","[452, 413]",16480
16496,"[413, 425]",8.360239,691,21,"[5.123622222222222, 3.2366166666666674]","[6017.5, 293, 347, 51]","[413, 425]","[425, 413]",16496


In [30]:
xyz = inData.sblts.schedule.copy()
xyz['pooling_reqs'] = xyz.apply(lambda x: any(i in travs_summary[travs_summary.chosen_mode == 'pool'].index.to_list() for i in x.indexes) and travs_summary.gets_offer.loc[x.indexes[0]], axis=1)
xyz[xyz.pooling_reqs]

,indexes,u_pax,u_veh,kind,u_paxes,times,indexes_orig,indexes_dest,index,lambda_r,PassHourTrav_ns,row,selected,degree,nodes,req_id,sim_schedule,pooling_reqs
21,[21],3.318444,259,1,[3.3184444444444448],"[228, 259]",[21],[21],21,0.350000,259,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, ...",1,1,"[None, 44861481, 3581094279]","[None, 21, 21]",node time req_id od 0 ...,True
87,[87],3.482556,272,1,[3.4825555555555554],"[1353, 272]",[87],[87],87,0.350000,272,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",1,1,"[None, 5507717333, 5279883912]","[None, 87, 87]",node time req_id od 0 ...,True
370,[370],6.089222,476,1,[6.089222222222222],"[5178, 476]",[370],[370],370,0.350000,476,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",1,1,"[None, 2047160810, 2381861256]","[None, 370, 370]",node time req_id od 0 ...,True
755,"[428, 454]",7.509978,804,20,"[3.3238666666666665, 4.186111111111112]","[6285.5, 368, 88, 348]","[428, 454]","[428, 454]",755,-0.135593,708,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",1,2,"[None, 1410537007, 1552651190, 1413910857, 447...","[None, 428, 454, 428, 454]",node time req_id od 0 ...,True
778,"[466, 470]",8.091839,804,20,"[3.7611333333333334, 4.330705555555555]","[6587.5, 323, 51, 430]","[466, 470]","[466, 470]",778,-0.066313,754,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",1,2,"[None, 5715066940, 44810363, 44820017, 44722748]","[None, 466, 470, 466, 470]",node time req_id od 0 ...,True
1507,"[298, 330]",6.928383,745,20,"[3.5033333333333334, 3.4250500000000006]","[4368.0, 309, 103, 333]","[298, 330]","[298, 330]",1507,-0.169545,637,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",1,2,"[None, 44777528, 44866496, 1371031395, 44774822]","[None, 298, 330, 298, 330]",node time req_id od 0 ...,True
2647,"[363, 398]",6.179350,673,20,"[4.11975, 2.0596]","[5117.5, 433, 50, 190]","[363, 398]","[363, 398]",2647,-0.087237,619,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",1,2,"[None, 2670677182, 44774832, 1393024898, 44825...","[None, 363, 398, 363, 398]",node time req_id od 0 ...,True
3306,"[174, 191]",4.516839,371,20,"[2.4866722222222224, 2.0301666666666667]","[2618.5, 143, 141, 87]","[174, 191]","[174, 191]",3306,0.182819,454,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",1,2,"[None, 4474353182, 44779745, 2321923795, 44840...","[None, 174, 191, 174, 191]",node time req_id od 0 ...,True
4336,"[186, 212]",7.220839,554,20,"[2.8292277777777777, 4.391611111111112]","[2917.5, 16, 249, 289]","[186, 212]","[186, 212]",4336,0.106452,620,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",1,2,"[None, 311106487, 44844082, 5264539946, 44861083]","[None, 186, 212, 186, 212]",node time req_id od 0 ...,True
4413,"[158, 179]",6.860822,546,20,"[3.5548, 3.3060222222222224]","[2447.0, 150, 256, 140]","[158, 179]","[158, 179]",4413,0.152174,644,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",1,2,"[None, 2572707863, 44847038, 44893325, 44863340]","[None, 158, 179, 158, 179]",node time req_id od 0 ...,True


In [82]:
sim.vehicles

,pos,event,shift_start,shift_end,platform,expected_income,res_wage,informed,registered,work_exp,days_since_reg,rejected_reg
veh,,,,,,,,,,,,
1,1571092666,driverEvent.ENDS_SHIFT,0,86400,0,27.382440,33.710874,True,True,2.0,0.0,False
2,1679761037,driverEvent.ENDS_SHIFT,0,86400,0,58.090460,78.609375,True,True,2.0,0.0,False
3,44763544,driverEvent.DECIDES_NOT_TO_DRIVE,0,86400,0,44.989374,53.222053,True,True,2.0,0.0,False
4,1385072819,driverEvent.ENDS_SHIFT,0,86400,0,92.943274,114.183968,True,True,1.0,0.0,False
5,5171875773,driverEvent.ENDS_SHIFT,0,86400,0,16.743354,13.270616,True,True,2.0,0.0,False
6,4301397894,driverEvent.ENDS_SHIFT,0,86400,0,34.657054,45.029428,True,True,2.0,0.0,False
7,1608998265,driverEvent.ENDS_SHIFT,0,86400,0,24.062915,26.309086,True,True,2.0,0.0,False
8,1435362416,driverEvent.ENDS_SHIFT,0,86400,0,35.958263,41.036454,True,True,1.0,0.0,False
9,4053473202,driverEvent.DECIDES_NOT_TO_DRIVE,0,86400,0,39.168261,39.168261,True,True,0.0,0.0,False


In [32]:
sim.last_res.veh_exp

,nRIDES,nREJECTED,DRIVING_TIME,DRIVING_DIST,REVENUE,COST,NET_INCOME,OUT,FORCED_OUT,STARTS_DAY,...,ARRIVES_AT_PICKUP,MEETS_TRAVELLER_AT_PICKUP,DEPARTS_FROM_PICKUP,ARRIVES_AT_DROPOFF,CONTINUES_SHIFT,STARTS_REPOSITIONING,REPOSITIONED,DECIDES_NOT_TO_DRIVE,ENDS_SHIFT,NOT_ALLOWED_TO_DRIVE
veh,,,,,,,,,,,,,,,,,,,,,
1,5.0,0.0,2063.0,20.63,20.8525,5.1575,15.6950,False,False,0,...,14.0,150.0,1695.0,11992.0,0,0,0,0,0,0
2,6.0,0.0,2251.0,22.51,23.0470,5.6275,17.4195,False,False,0,...,0.0,180.0,1826.0,10185.0,0,0,0,0,0,0
3,0.0,0.0,0.0,0.00,0.0000,0.0000,0.0000,True,False,0,...,0.0,0.0,0.0,0.0,0,0,0,0,0,0
4,3.0,0.0,1611.0,16.11,12.0080,4.0275,7.9805,False,False,0,...,0.0,90.0,964.0,11882.0,0,0,0,0,0,0
5,7.0,0.0,2811.0,28.11,28.1295,7.0275,21.1020,False,False,0,...,16.0,210.0,2261.0,10107.0,0,0,0,0,0,0
6,4.0,0.0,1504.0,15.04,13.1955,3.7600,9.4355,False,False,0,...,0.0,120.0,989.0,11607.0,0,0,0,0,0,0
7,4.0,0.0,1215.0,12.15,14.4970,3.0375,11.4595,False,False,0,...,51.0,120.0,1126.0,11157.0,0,0,0,0,0,0
8,6.0,0.0,1965.0,19.65,20.5580,4.9125,15.6455,False,False,0,...,0.0,180.0,1564.0,11196.0,0,0,0,0,0,0
9,0.0,0.0,0.0,0.00,0.0000,0.0000,0.0000,True,False,0,...,0.0,0.0,0.0,0.0,0,0,0,0,0,0


In [33]:
travs_summary

,orig,dest,t_req,tt_min,dist,informed,requests,gets_offer,accepts_offer,xp_wait,...,new_perc_wait,new_perc_wait_pool,chosen_mode,act_shared,xp_discount,xp_detour,init_perc_disc,new_perc_disc,init_perc_detour,new_perc_detour
pax,,,,,,,,,,,,,,,,,,,,,
0,1679761158,2612573901,2021-11-01 08:00:09,0 days 00:04:22,2623,True,True,True,True,99.0,...,19.8,NaN,rs,False,NaN,0.0,0.36,0.36,43.6,43.6
1,44824666,44764023,2021-11-01 08:00:20,0 days 00:05:54,3545,True,False,False,False,NaN,...,6.0,NaN,bike,False,NaN,NaN,0.36,0.36,65.2,65.2
2,4053473202,1385072823,2021-11-01 08:00:23,0 days 00:04:01,2416,True,False,False,False,NaN,...,5.0,NaN,bike,False,NaN,NaN,0.30,0.30,0.0,0.0
3,44862225,44759522,2021-11-01 08:00:38,0 days 00:06:38,3983,True,False,False,False,NaN,...,27.0,NaN,bike,False,NaN,NaN,0.30,0.30,0.0,0.0
4,1679761151,44839603,2021-11-01 08:00:54,0 days 00:03:32,2125,True,False,False,False,NaN,...,0.0,7.8,car,False,NaN,NaN,0.36,0.36,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,1413911080,2321923795,2021-11-01 09:58:38,0 days 00:04:14,2549,True,False,False,False,NaN,...,0.0,NaN,bike,False,NaN,NaN,0.30,0.30,0.0,0.0
496,1421472391,1519889948,2021-11-01 09:58:56,0 days 00:03:57,2374,True,False,False,False,NaN,...,17.0,NaN,bike,False,NaN,NaN,0.30,0.30,0.0,0.0
497,44817967,1679761035,2021-11-01 09:59:02,0 days 00:07:39,4590,True,False,False,False,NaN,...,13.6,NaN,bike,False,NaN,NaN,0.30,0.30,0.0,0.0


In [72]:
user_stats = pd.read_csv('FleetPy_res/1_user-stats.csv') # request LoS indicators
veh_types = pd.read_csv('FleetPy_res/2_vehicle_types.csv') # vehicle type per driver (empty)
op_stats = pd.read_csv('FleetPy_res/2-2_op-stats.csv') # operational statistics for a single ride
op_dyn_atts = pd.read_csv('FleetPy_res/3-0_op-dyn_atts.csv') #??
final_state = pd.read_csv('FleetPy_res/final_state.csv',index_col=0) # final state of vehicles
standard_eval = pd.read_csv('FleetPy_res/standard_eval.csv') # Platform KPIs
standard_mod_0 = pd.read_csv('FleetPy_res/standard_mod-0_veh_eval.csv',index_col=0) # KPIs per driver (working for platform 0)
standard_mod_1 = pd.read_csv('FleetPy_res/standard_mod-1_veh_eval.csv',index_col=0) # KPIs per driver (working for platform 1)

In [81]:
# pd.set_option('display.max_columns', None)
standard_mod_1

,type,total km,total kWh,total CO2 [g],fix costs,total variable costs,revenue,driver_id,km occ 0,km occ 1,km occ 2
8,default_vehtype,33.732732,0.168664,18.890330,2500.0,843.0,675.00,9,8.943683,23.570297,1.218752
9,default_vehtype,23.178157,0.115891,12.979768,2500.0,579.0,518.25,10,5.940422,15.389093,1.848641
7,default_vehtype,21.368141,0.106841,11.966159,2500.0,534.0,340.75,8,8.711455,12.656687,NaN
3,default_vehtype,17.190738,0.085954,9.626813,2500.0,430.0,312.00,4,5.872775,11.317964,NaN
2,default_vehtype,8.957085,0.044785,5.015968,2500.0,224.0,194.50,3,2.141035,6.723941,0.092109
1,default_vehtype,7.063771,0.035319,3.955712,2500.0,177.0,143.75,2,1.519409,5.544362,NaN
0,default_vehtype,3.315610,0.016578,1.856742,2500.0,83.0,62.75,1,0.796109,2.519501,NaN
